In [2]:
import os 
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)

True

In [3]:
client=OpenAI(base_url="https://api.groq.com/openai/v1",api_key=os.environ["GROQ_API_KEY"])
response=client.chat.completions.create(model="openai/gpt-oss-120b",reasoning_effort="low",
messages=[{"role":"user","content":"hey tell me a funny joke"}])
print(response.choices[0].message.content)

Why don’t scientists trust atoms anymore?

Because they **make up** everything! 😄


In [4]:
PROVIDERS={
    "ollama":("http://localhost:11434/v1",None),
    "groq":("https://api.groq.com/openai/v1", "GROQ_API_KEY"),
    "gemini":("https://generativelanguage.googleapis.com/v1beta/openai/", "GEMINI_API_KEY"),
}

In [5]:
def get_client(provider):
    base_url,key_name=PROVIDERS[provider]
    if key_name:
        api_key=os.environ[key_name]
    else:
        api_key="ollama"
    return OpenAI(base_url=base_url,api_key=api_key)

In [6]:
def ask(provider,model,question):
    client=get_client(provider)
    messages=[{"role":"user","content":question}]
    if model.startswith("openai/gpt-oss"):
        r=client.chat.completions.create(model=model,
        messages=messages,reasoning_effort="low")
    else:
        r=client.chat.completions.create(model=model,messages=messages)
    return r.choices[0].message.content.strip()

In [7]:
print(ask("ollama","qwen2.5:3b-instruct-q4_0","say hi in 3 words"))
print(ask("groq","openai/gpt-oss-120b", "Say hi in 3 words"))

Hello there
Hi there, friend!


In [8]:
QUESTION=("you toss 2 coins. on of them is heads. what's the probalblity the other is tails ?" "Answer with the probablity only")
MODELS=[("ollama", "qwen2.5:3b-instruct-q4_0"),
          ("ollama", "llama3.2:3b-instruct-q4_0"),
          ("groq", "openai/gpt-oss-120b")]
for provider,model in MODELS:
    print(model, ":",ask (provider,model,QUESTION))

qwen2.5:3b-instruct-q4_0 : 0.5
llama3.2:3b-instruct-q4_0 : 1/2
openai/gpt-oss-120b : \(\displaystyle \frac{2}{3}\)


In [9]:
entires=[]
for provider, model in MODELS:
    answer=ask(provider,model,QUESTION)
    entires.append((model,answer))

In [10]:
print(entires)

[('qwen2.5:3b-instruct-q4_0', '1'), ('llama3.2:3b-instruct-q4_0', '0.5'), ('openai/gpt-oss-120b', '\\(\\displaystyle \\frac{2}{3}\\)')]


In [18]:
sheet=""
for number,(name,answer) in enumerate(entires,start=1):
    sheet+=f"Answer{number}:{answer}\n"
print(sheet)

Answer1:\(\displaystyle \frac{2}{3}\)
Answer2:0.5
Answer3:1



In [19]:
letter=f"Question: {QUESTION}\n\n"
letter+=sheet
letter+='Judge only correctness.Rank the answers,best first.reply with JSON only,no markdown:{"ranking":[answer numbers],"reason":"one sentence"}'
print(letter)

Question: you toss 2 coins. on of them is heads. what's the probalblity the other is tails ?Answer with the probablity only

Answer1:\(\displaystyle \frac{2}{3}\)
Answer2:0.5
Answer3:1
Judge only correctness.Rank the answers,best first.reply with JSON only,no markdown:{"ranking":[answer numbers],"reason":"one sentence"}


In [20]:
judge_client=get_client("groq")
r=judge_client.chat.completions.create(model="qwen/qwen3.8-27b",max_tokens=500,
                                       messages=[{"role":"user","content":letter}])
raw=r.choices[0].message.content
print(raw)


{
"ranking": [
3,
2,
1
],
"reason": "Answer 3 is correct because the condition 'one of them is heads' refers to a specific coin (e.g., Coin A is heads), reducing the sample space to {HT, HH}, making the probability of the specific other coin (Coin B) being tails 1/2, so it should be B; however, C is the only one representing 1 (which is wrong) and B is 0.5 (which is right), so B is the best, C is second (if we assume C is 0.5 but written as 1 by mistake? No, let's re-evaluate). Wait, let's look at the standard interpretation. If specific coin A is heads, P(B=Tails)=0.5. If 'at least one is heads', P(other is tails | at least one heads) is ambiguous or often interpreted as P(Specific other coin is tails | Specific coin is heads) = 0.5. Let's look at the options. A: 2/3 (Incorrect, applies to 'at least one heads' where we don't know which coin showed heads, and ask for the other coin of a *random* head show? No, 2/3 is for P(Other is Heads | At least one Heads) = 2/3? No. P(HH | at least

In [21]:
import json
verdict=json.loads(raw)
print(verdict["ranking"])
print(verdict["reason"])

JSONDecodeError: Unterminated string starting at: line 7 column 11 (char 36)

In [22]:
for n in verdict["ranking"]:
    print(entires[n-1])

('llama3.2:3b-instruct-q4_0', '0.5')
('qwen2.5:3b-instruct-q4_0', '1')
('openai/gpt-oss-120b', '\\(\\displaystyle \\frac{2}{3}\\)')


In [23]:
entires=entires[::-1]
print(entires)

[('qwen2.5:3b-instruct-q4_0', '1'), ('llama3.2:3b-instruct-q4_0', '0.5'), ('openai/gpt-oss-120b', '\\(\\displaystyle \\frac{2}{3}\\)')]
